# Stage 3: Sentiment-Only Training (Frozen Encoder + NER)

Loads the Stage 2 checkpoint, freezes the encoder and NER head, and trains **only the sentiment head** on relabeled data.

**Why:** Stage 2 sentiment Pearson correlation was 0.1198 — very poor. All train/val data has been relabeled with consistent Sonnet-generated sentiment scores. This stage fine-tunes only the sentiment head (~530K params) while preserving the encoder and NER head exactly.

**Model:** `allenai/longformer-large-4096` (1024-dim, CRF NER)

**Target:** Sentiment correlation > 0.4

**Data (relabeled):**
- Training: `data/labeled/final/train.jsonl` (17,593 articles)
- Validation: `data/labeled/final/val.jsonl` (2,130 articles)
- Holdout: `data/labeled/final/holdout.jsonl` (9,371 articles, original labels)

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set project path
import os

PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"

assert os.path.exists(PROJECT_PATH), f"Project path not found: {PROJECT_PATH}\nRun: !ls -la /content/drive/ to see available paths"
print(f"Project found at: {PROJECT_PATH}")

In [ ]:
# Install dependencies
!pip install -q transformers torch torchvision torchaudio
!pip install -q pytorch-crf
!pip install -q accelerate

In [ ]:
# Authenticate with Hugging Face Hub
import os
from getpass import getpass

hf_token = None

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("Found HF_TOKEN in Colab Secrets")
except (ImportError, userdata.SecretNotFoundError):
    print("HF_TOKEN not found in Colab Secrets.")
    hf_token = getpass("Paste your HF token (or press Enter to skip): ").strip()
    if not hf_token:
        print("Skipping HF auth")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("Authenticated with Hugging Face Hub")

In [ ]:
# Add project to Python path
import sys
sys.path.insert(0, PROJECT_PATH)

os.chdir(PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")

## 2. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime

# Stage 3: Sentiment-only training on relabeled data
CONFIG = {
    # Data — relabeled with Sonnet (replaced in-place)
    "train_file": f"{PROJECT_PATH}/data/labeled/final/train.jsonl",
    "val_file": f"{PROJECT_PATH}/data/labeled/final/val.jsonl",
    "holdout_file": f"{PROJECT_PATH}/data/labeled/final/holdout.jsonl",

    # Model — must match Stage 2 architecture
    "encoder_name": "allenai/longformer-large-4096",
    "hidden_size": 1024,
    "max_length": 2048,
    "use_crf": True,

    # Checkpoint paths
    "stage2_checkpoint": f"{PROJECT_PATH}/checkpoints/stage2_joint_large/best_model.pt",
    "stage3_checkpoint_dir": f"{PROJECT_PATH}/checkpoints/stage3_sentiment_large",

    # Training — memory budget on A100 80GB:
    #   batch=32 OOMs (~79GB), batch=8 eval=9.3GB
    #   ~2.4 GB/sample during training (encoder activations retained in graph)
    #   batch=24 → ~60GB estimated, safe headroom
    "batch_size": 52,
    "gradient_accumulation": 1,    # effective batch = 50
    "epochs": 10,
    "lr": 5e-4,                    # higher LR for small head
    "weight_decay": 0.01,
    "gradient_clip": 1.0,

    # Loss weights — sentiment only
    "ner_weight": 0.0,
    "sentiment_weight": 1.0,

    # Early stopping
    "patience": 3,                 # stop if no improvement for 3 epochs
}

os.makedirs(CONFIG["stage3_checkpoint_dir"], exist_ok=True)

print("Stage 3 Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nEffective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation']}")


## 3. Load Data

In [ ]:
import torch
from training.preprocessing import DataPreprocessor
from training.dataset import create_data_loaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Verify relabeled data exists
for label, path in [("Train", CONFIG["train_file"]), ("Val", CONFIG["val_file"])]:
    assert os.path.exists(path), f"{label} file not found: {path}"
print("\nData files found:")
!wc -l "{CONFIG['train_file']}" "{CONFIG['val_file']}" "{CONFIG['holdout_file']}"

In [ ]:
# Initialize preprocessor and load data
preprocessor = DataPreprocessor(
    model_name=CONFIG["encoder_name"],
    max_length=CONFIG["max_length"],
)

print("Loading and preprocessing relabeled data...")
train_loader, val_loader = create_data_loaders(
    train_files=CONFIG["train_file"],
    val_files=CONFIG["val_file"],
    preprocessor=preprocessor,
    batch_size=CONFIG["batch_size"],
)

print(f"\nData loaded:")
print(f"  Train batches: {len(train_loader)} (batch_size={CONFIG['batch_size']})")
print(f"  Val batches: {len(val_loader)}")
print(f"  Train samples: ~{len(train_loader) * CONFIG['batch_size']}")
print(f"  Val samples: ~{len(val_loader) * CONFIG['batch_size']}")

## 4. Load Model & Freeze

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from models.pipeline import FinancialEntitySentimentModel

print("="*60)
print("STAGE 3: SENTIMENT-ONLY TRAINING (V2 Cross-Attention Head)")
print("="*60)

# Create model with V2 sentiment head (fresh init via Xavier)
model = FinancialEntitySentimentModel(
    encoder_name=CONFIG["encoder_name"],
    hidden_size=CONFIG["hidden_size"],
    use_crf_ner=CONFIG["use_crf"],
)
model = model.to(device)

# Load Stage 2 checkpoint (trained with v1 sentiment head)
# Discard old sentiment_head.* weights — V2 architecture is different
print(f"\nLoading Stage 2 checkpoint: {CONFIG['stage2_checkpoint']}")
checkpoint = torch.load(CONFIG["stage2_checkpoint"], map_location=device, weights_only=False)
stage2_state = checkpoint["model_state_dict"]

# Filter out old v1 sentiment head weights
v1_sentiment_keys = [k for k in stage2_state if k.startswith("sentiment_head.")]
stage2_state = {k: v for k, v in stage2_state.items() if not k.startswith("sentiment_head.")}
model.load_state_dict(stage2_state, strict=False)

stage2_epoch = checkpoint.get("epoch", -1) + 1
stage2_metrics = checkpoint.get("val_metrics", {})
print(f"Stage 2 encoder + NER weights loaded (from epoch {stage2_epoch})")
print(f"  Discarded {len(v1_sentiment_keys)} v1 sentiment head params: {v1_sentiment_keys}")
print(f"  V2 sentiment head initialized fresh (Xavier init)")
if stage2_metrics:
    print(f"  Stage 2 NER F1:          {stage2_metrics.get('ner_f1', 'N/A')}")
    print(f"  Stage 2 Sentiment MSE:   {stage2_metrics.get('sentiment_mse', 'N/A')}")
    print(f"  Stage 2 Sentiment Corr:  {stage2_metrics.get('sentiment_corr', 'N/A')}")
del checkpoint
torch.cuda.empty_cache()


In [ ]:
# Freeze encoder
model.freeze_encoder()

# Additionally freeze the NER head — we only want sentiment head trainable
for name, param in model.named_parameters():
    if name.startswith("ner_head."):
        param.requires_grad = False

# Verify: count trainable vs total parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
frozen_params = total_params - trainable_params

print(f"\nParameter summary:")
print(f"  Total:     {total_params:>12,}")
print(f"  Frozen:    {frozen_params:>12,}")
print(f"  Trainable: {trainable_params:>12,}  ({100*trainable_params/total_params:.2f}%)")

# Show which modules are trainable
print(f"\nTrainable modules:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.numel():,} params")

## 5. Training Loop

In [ ]:
import time
from tqdm.auto import tqdm
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from training.trainer import compute_ner_metrics, compute_sentiment_metrics


def train_epoch_with_accumulation(
    model,
    train_loader,
    optimizer,
    scheduler,
    scaler,
    device,
    accumulation_steps,
    ner_weight=0.0,
    sentiment_weight=1.0,
    gradient_clip=1.0,
):
    """Train for one epoch with gradient accumulation."""
    model.train()

    total_loss = 0.0
    total_ner_loss = 0.0
    total_sentiment_loss = 0.0
    num_batches = 0

    ner_criterion = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)

    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc="Training")
    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        ner_labels = batch["ner_labels"].to(device)
        entity_masks = batch["entity_masks"].to(device)
        sentiment_targets = batch["sentiment_scores"].to(device)
        entity_mask_valid = batch["entity_mask_valid"].to(device)

        # Forward pass in float16 (autocast)
        with torch.amp.autocast(device_type="cuda"):
            ner_output, sentiment_preds = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                entity_masks=entity_masks,
                ner_labels=ner_labels,
            )

            # NER loss (safe in float16)
            if isinstance(ner_output, dict):
                ner_loss = ner_output.get("loss", torch.tensor(0.0, device=device))
            else:
                batch_size_cur, seq_len, num_labels = ner_output.shape
                ner_loss = ner_criterion(
                    ner_output.view(-1, num_labels),
                    ner_labels.view(-1)
                )

        # Sentiment loss in float32 — Pearson correlation underflows in float16
        valid_mask = entity_mask_valid.bool()
        valid_preds = sentiment_preds[valid_mask].float()
        valid_targets = sentiment_targets[valid_mask].float()
        if valid_preds.numel() > 0:
            sentiment_loss = model.sentiment_head.compute_loss(valid_preds, valid_targets)
        else:
            sentiment_loss = torch.tensor(0.0, device=device)

        loss = (ner_weight * ner_loss.float() + sentiment_weight * sentiment_loss) / accumulation_steps

        scaler.scale(loss).backward()

        total_loss += loss.item() * accumulation_steps
        total_ner_loss += ner_loss.item()
        total_sentiment_loss += sentiment_loss.item() if valid_preds.numel() > 0 else 0
        num_batches += 1

        if (step + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

            pbar.set_postfix({
                "loss": f"{total_loss/num_batches:.4f}",
                "sent": f"{total_sentiment_loss/num_batches:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}"
            })

    # Handle remaining gradients
    if (step + 1) % accumulation_steps != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    return {
        "train_loss": total_loss / num_batches,
        "train_ner_loss": total_ner_loss / num_batches,
        "train_sentiment_loss": total_sentiment_loss / num_batches,
    }


@torch.no_grad()
def evaluate_model(model, val_loader, device):
    """Evaluate model on validation set."""
    model.eval()

    ner_criterion = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)

    total_ner_loss = 0.0
    total_sentiment_loss = 0.0
    total_samples = 0
    total_entities = 0

    all_ner_preds = []
    all_ner_labels = []
    all_sentiment_preds = []
    all_sentiment_targets = []

    for batch in tqdm(val_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        ner_labels = batch["ner_labels"].to(device)
        entity_masks = batch["entity_masks"].to(device)
        sentiment_targets = batch["sentiment_scores"].to(device)
        entity_mask_valid = batch["entity_mask_valid"].to(device)

        batch_size = input_ids.shape[0]

        with torch.amp.autocast(device_type="cuda"):
            ner_output, sentiment_preds = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                entity_masks=entity_masks,
            )

        # Cast sentiment preds to float32 for stable metric computation
        sentiment_preds = sentiment_preds.float()

        # NER loss
        if isinstance(ner_output, dict):
            ner_logits = ner_output["logits"]
            if hasattr(model.ner_head, 'crf'):
                ner_loss = -model.ner_head.crf(
                    ner_logits, ner_labels,
                    mask=attention_mask.bool(), reduction='mean'
                )
            else:
                _, seq_len, num_labels = ner_logits.shape
                ner_loss = ner_criterion(
                    ner_logits.view(-1, num_labels),
                    ner_labels.view(-1),
                )
        else:
            ner_logits = ner_output
            _, seq_len, num_labels = ner_logits.shape
            ner_loss = ner_criterion(
                ner_logits.view(-1, num_labels),
                ner_labels.view(-1),
            )
        total_ner_loss += ner_loss.item() * batch_size

        # Sentiment loss — V2 combined loss for eval tracking
        valid_mask = entity_mask_valid.bool()
        valid_preds = sentiment_preds[valid_mask]
        valid_targets = sentiment_targets[valid_mask]
        num_valid = valid_preds.numel()
        if num_valid > 0:
            sentiment_loss = model.sentiment_head.compute_loss(valid_preds, valid_targets)
            total_sentiment_loss += sentiment_loss.item() * num_valid
            total_entities += num_valid

        total_samples += batch_size

        # Collect NER predictions
        if isinstance(ner_output, dict) and "predictions" in ner_output:
            ner_preds = ner_output["predictions"]
        else:
            ner_preds = ner_logits.argmax(dim=-1)

        valid_tok_mask = attention_mask.bool()
        for i in range(batch_size):
            valid_tokens = valid_tok_mask[i]
            all_ner_preds.extend(ner_preds[i][valid_tokens].cpu().tolist())
            all_ner_labels.extend(ner_labels[i][valid_tokens].cpu().tolist())

        # Collect sentiment predictions
        for i in range(batch_size):
            for j in range(entity_mask_valid.shape[1]):
                if entity_mask_valid[i, j] > 0:
                    all_sentiment_preds.append(sentiment_preds[i, j].item())
                    all_sentiment_targets.append(sentiment_targets[i, j].item())

    # Compute metrics
    metrics = {
        "val_ner_loss": total_ner_loss / total_samples,
        "val_sentiment_loss": total_sentiment_loss / max(total_entities, 1),
        "val_total_loss": (
            total_ner_loss / total_samples +
            0.5 * total_sentiment_loss / max(total_entities, 1)
        ),
    }

    ner_metrics = compute_ner_metrics(all_ner_preds, all_ner_labels)
    metrics.update(ner_metrics)

    if all_sentiment_preds:
        sentiment_metrics = compute_sentiment_metrics(all_sentiment_preds, all_sentiment_targets)
        metrics.update(sentiment_metrics)

    return metrics


print("Training functions defined.")


In [ ]:
# Setup logging
LOG_FILE = f"{PROJECT_PATH}/logs/stage3_sentiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
os.makedirs(f"{PROJECT_PATH}/logs", exist_ok=True)

def log_message(msg):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_FILE, "a") as f:
        f.write(f"{timestamp} - {msg}\n")
    print(msg)

# Only optimize sentiment head parameters (the ones with requires_grad=True)
trainable_params_list = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params_list, lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])

num_steps = len(train_loader) * CONFIG["epochs"]
scheduler = CosineAnnealingLR(optimizer, T_max=num_steps)
scaler = torch.amp.GradScaler()

log_message("="*60)
log_message("STAGE 3: SENTIMENT-ONLY TRAINING")
log_message("="*60)
log_message(f"Trainable params: {sum(p.numel() for p in trainable_params_list):,}")
log_message(f"Config: batch={CONFIG['batch_size']}x{CONFIG['gradient_accumulation']}, lr={CONFIG['lr']}, epochs={CONFIG['epochs']}")
log_message(f"Loss weights: ner={CONFIG['ner_weight']}, sentiment={CONFIG['sentiment_weight']}")
log_message(f"Early stopping patience: {CONFIG['patience']}")
log_message(f"Optimizer steps: {num_steps}")
log_message(f"Log file: {LOG_FILE}")
if torch.cuda.is_available():
    log_message(f"GPU: {torch.cuda.get_device_name(0)}")
    log_message(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

In [ ]:
# Evaluate Stage 2 baseline on relabeled val data before training
log_message("\nEvaluating Stage 2 baseline on relabeled val data...")
baseline_metrics = evaluate_model(model, val_loader, device)
log_message(f"Stage 2 baseline (relabeled val):")
log_message(f"  NER F1:          {baseline_metrics.get('ner_f1', 0):.4f}")
log_message(f"  Sentiment MSE:   {baseline_metrics.get('sentiment_mse', 0):.4f}")
log_message(f"  Sentiment MAE:   {baseline_metrics.get('sentiment_mae', 0):.4f}")
log_message(f"  Sentiment Corr:  {baseline_metrics.get('sentiment_corr', 0):.4f}")

In [ ]:
# Training loop — select best model by sentiment_corr
log_message("\nStarting Stage 3 training...")
start_time = time.time()

best_sentiment_corr = baseline_metrics.get("sentiment_corr", -1.0)
patience_counter = 0

history = {
    "train_loss": [],
    "train_sentiment_loss": [],
    "val_sentiment_mse": [],
    "val_sentiment_mae": [],
    "val_sentiment_corr": [],
    "val_ner_f1": [],
}

accumulation_steps = CONFIG["gradient_accumulation"]

for epoch in range(CONFIG["epochs"]):
    epoch_start = time.time()

    log_message(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")

    # Train
    train_metrics = train_epoch_with_accumulation(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=device,
        accumulation_steps=accumulation_steps,
        ner_weight=CONFIG["ner_weight"],
        sentiment_weight=CONFIG["sentiment_weight"],
        gradient_clip=CONFIG["gradient_clip"],
    )

    # Evaluate
    val_metrics = evaluate_model(model, val_loader, device)

    epoch_time = time.time() - epoch_start
    gpu_mem = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0

    # Record history
    history["train_loss"].append(train_metrics["train_loss"])
    history["train_sentiment_loss"].append(train_metrics["train_sentiment_loss"])
    history["val_sentiment_mse"].append(val_metrics.get("sentiment_mse", 0))
    history["val_sentiment_mae"].append(val_metrics.get("sentiment_mae", 0))
    history["val_sentiment_corr"].append(val_metrics.get("sentiment_corr", 0))
    history["val_ner_f1"].append(val_metrics.get("ner_f1", 0))

    current_corr = val_metrics.get("sentiment_corr", 0)

    log_message(
        f"Epoch {epoch + 1}: "
        f"train_loss={train_metrics['train_loss']:.4f}, "
        f"sent_mse={val_metrics.get('sentiment_mse', 0):.4f}, "
        f"sent_mae={val_metrics.get('sentiment_mae', 0):.4f}, "
        f"sent_corr={current_corr:.4f}, "
        f"ner_f1={val_metrics.get('ner_f1', 0):.4f}, "
        f"time={epoch_time/60:.1f}min, "
        f"gpu_peak={gpu_mem:.1f}GB"
    )

    # Best model by sentiment correlation
    if current_corr > best_sentiment_corr:
        best_sentiment_corr = current_corr
        patience_counter = 0
        local_path = "/content/best_model_backup.pt"
        drive_path = f"{CONFIG['stage3_checkpoint_dir']}/best_model.pt"
        ckpt_data = {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_metrics": val_metrics,
            "epoch": epoch,
            "history": history,
        }
        # Save LOCAL first (fast, reliable)
        torch.save(ckpt_data, local_path)
        ckpt_size = os.path.getsize(local_path) / 1e6
        log_message(f"  -> New best model! sentiment_corr={best_sentiment_corr:.4f} (saved {ckpt_size:.0f} MB locally)")
        # Then copy to Drive (may be slow but local is safe)
        import shutil
        try:
            shutil.copy2(local_path, drive_path)
            log_message(f"     Copied to Drive: {drive_path}")
        except Exception as e:
            log_message(f"     WARNING: Drive copy failed ({e}), local backup OK at {local_path}")
    else:
        patience_counter += 1
        log_message(f"  -> No improvement ({patience_counter}/{CONFIG['patience']})")

    # Save periodic checkpoint — local first, then Drive
    local_epoch_path = f"/content/checkpoint_epoch_{epoch+1}.pt"
    drive_epoch_path = f"{CONFIG['stage3_checkpoint_dir']}/checkpoint_epoch_{epoch+1}.pt"
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "epoch": epoch,
        "history": history,
    }, local_epoch_path)
    try:
        import shutil
        shutil.copy2(local_epoch_path, drive_epoch_path)
    except Exception as e:
        log_message(f"     WARNING: Drive copy for epoch {epoch+1} failed ({e})")

    # Early stopping
    if patience_counter >= CONFIG["patience"]:
        log_message(f"\nEarly stopping at epoch {epoch + 1} (no improvement for {CONFIG['patience']} epochs)")
        break

    # ETA
    elapsed = time.time() - start_time
    time_per_epoch = elapsed / (epoch + 1)
    remaining = time_per_epoch * (CONFIG["epochs"] - (epoch + 1))
    log_message(f"  -> ETA: {remaining/60:.1f} min remaining")

elapsed = time.time() - start_time
log_message("\n" + "="*60)
log_message("STAGE 3 COMPLETE!")
log_message(f"Total time: {elapsed/60:.1f} min ({elapsed/3600:.2f} hours)")
log_message(f"Best sentiment correlation: {best_sentiment_corr:.4f}")
log_message("="*60)

# Flush Drive to ensure checkpoints are synced
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("Drive flushed. Remounting...")
    drive.mount("/content/drive")
    print("Drive remounted. Checkpoints should now be synced.")
except Exception as e:
    print(f"Drive flush warning: {e}")

# Verify checkpoints exist and are non-trivial
print("\nCheckpoint verification:")
for f in sorted(os.listdir(CONFIG["stage3_checkpoint_dir"])):
    path = os.path.join(CONFIG["stage3_checkpoint_dir"], f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f:35s} {size_mb:8.1f} MB")


## 6. Training Summary & Visualization

In [ ]:
import matplotlib.pyplot as plt

num_epochs_trained = len(history["train_loss"])
epochs_x = range(1, num_epochs_trained + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sentiment MSE
ax = axes[0, 0]
ax.plot(epochs_x, history["train_sentiment_loss"], 'b-', label='Train MSE')
ax.plot(epochs_x, history["val_sentiment_mse"], 'r-', label='Val MSE')
ax.axhline(y=baseline_metrics.get('sentiment_mse', 0), color='gray', linestyle='--', label='Stage 2 baseline')
ax.set_title('Sentiment MSE')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.legend()

# Sentiment Correlation
ax = axes[0, 1]
ax.plot(epochs_x, history["val_sentiment_corr"], 'g-o', label='Val Correlation', linewidth=2)
ax.axhline(y=baseline_metrics.get('sentiment_corr', 0), color='gray', linestyle='--', label='Stage 2 baseline')
ax.axhline(y=0.4, color='red', linestyle=':', alpha=0.5, label='Target (0.4)')
ax.set_title('Sentiment Pearson Correlation')
ax.set_xlabel('Epoch')
ax.set_ylabel('Correlation')
ax.legend()

# NER F1 (should stay constant)
ax = axes[1, 0]
ax.plot(epochs_x, history["val_ner_f1"], 'g-o', label='NER F1')
ax.axhline(y=baseline_metrics.get('ner_f1', 0), color='gray', linestyle='--', label='Stage 2 baseline')
ax.set_title('NER F1 (should be unchanged)')
ax.set_xlabel('Epoch')
ax.set_ylabel('F1 Score')
ax.legend()

# Summary
ax = axes[1, 1]
ax.axis('off')
baseline_corr = baseline_metrics.get('sentiment_corr', 0)
summary_text = f"""
STAGE 3 SUMMARY
==================================

Frozen: encoder + NER head
Trainable: sentiment head only
  ({trainable_params:,} params)

Training: {num_epochs_trained} epochs
LR: {CONFIG['lr']}, Batch: {CONFIG['batch_size']}

Sentiment Correlation:
  Stage 2 baseline: {baseline_corr:.4f}
  Stage 3 best:     {best_sentiment_corr:.4f}
  Improvement:      {best_sentiment_corr - baseline_corr:+.4f}

NER F1:
  Stage 2: {baseline_metrics.get('ner_f1', 0):.4f}
  Stage 3: {history['val_ner_f1'][-1]:.4f}
"""
ax.text(0.05, 0.5, summary_text, fontsize=11, family='monospace',
        verticalalignment='center', transform=ax.transAxes)

plt.tight_layout()
os.makedirs(f"{PROJECT_PATH}/outputs", exist_ok=True)
plt.savefig(f"{PROJECT_PATH}/outputs/stage3_training_curves.png", dpi=150)
plt.show()
print(f"\nPlot saved to: {PROJECT_PATH}/outputs/stage3_training_curves.png")

## 7. Holdout Evaluation

In [ ]:
import pickle

# Load best Stage 3 model for holdout evaluation
best_ckpt_path = f"{CONFIG['stage3_checkpoint_dir']}/best_model.pt"
local_backup = "/content/best_model_backup.pt"
best_ckpt = None

# Try to load checkpoint, with fallback to local backup if corrupted
def try_load_checkpoint(path, device):
    """Try to load checkpoint, return None if corrupted."""
    if not os.path.exists(path):
        return None
    try:
        ckpt = torch.load(path, map_location=device, weights_only=False)
        # Verify it has required keys
        if "model_state_dict" not in ckpt:
            print(f"Warning: Checkpoint {path} missing 'model_state_dict'")
            return None
        return ckpt
    except (RuntimeError, EOFError, pickle.UnpicklingError) as e:
        print(f"Warning: Failed to load checkpoint from {path}: {e}")
        return None

# Try Drive checkpoint first
best_ckpt = try_load_checkpoint(best_ckpt_path, device)
if best_ckpt:
    print(f"Loaded best Stage 3 model from Drive")
else:
    # Try local backup
    best_ckpt = try_load_checkpoint(local_backup, device)
    if best_ckpt:
        print(f"Drive checkpoint corrupted/missing, using local backup: {local_backup}")
    else:
        print("WARNING: No valid checkpoint found. Using last epoch weights from training.")

if best_ckpt:
    model.load_state_dict(best_ckpt["model_state_dict"])
    print(f"  Epoch: {best_ckpt.get('epoch', -1) + 1}")
    print(f"  Val Correlation: {best_ckpt.get('val_metrics', {}).get('sentiment_corr', 0):.4f}")
    del best_ckpt
    torch.cuda.empty_cache()


# Evaluate on holdout
if os.path.exists(CONFIG["holdout_file"]):
    print("\nPreprocessing holdout set...")
    holdout_samples = preprocessor.process_file(CONFIG["holdout_file"])
    print(f"Preprocessed {len(holdout_samples)} holdout samples")

    from torch.utils.data import DataLoader
    from training.dataset import EntitySentimentDataset, collate_fn
    holdout_dataset = EntitySentimentDataset(holdout_samples, max_entities_per_sample=10)
    holdout_loader = DataLoader(
        holdout_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )
    print(f"Holdout DataLoader: {len(holdout_loader)} batches")

    holdout_metrics = evaluate_model(model, holdout_loader, device)

    print("\n" + "="*60)
    print(f"HOLDOUT EVALUATION ({len(holdout_samples)} articles)")
    print("="*60)
    print(f"  NER F1:            {holdout_metrics.get('ner_f1', 0):.4f}")
    print(f"  NER Precision:     {holdout_metrics.get('ner_precision', 0):.4f}")
    print(f"  NER Recall:        {holdout_metrics.get('ner_recall', 0):.4f}")
    print(f"  Sentiment MSE:     {holdout_metrics.get('sentiment_mse', 0):.4f}")
    print(f"  Sentiment MAE:     {holdout_metrics.get('sentiment_mae', 0):.4f}")
    print(f"  Sentiment Corr:    {holdout_metrics.get('sentiment_corr', 0):.4f}")
    print("="*60)

    # Log holdout results
    log_message("\nHOLDOUT EVALUATION:")
    for k, v in holdout_metrics.items():
        log_message(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
else:
    print(f"Holdout file not found: {CONFIG['holdout_file']}")


## 8. Checkpoint Summary

In [ ]:
print("="*60)
print("STAGE 3 TRAINING COMPLETE!")
print("="*60)

print(f"\nModel checkpoints:")
print(f"  Stage 2: {CONFIG['stage2_checkpoint']}")
print(f"  Stage 3: {CONFIG['stage3_checkpoint_dir']}/best_model.pt")

print(f"\nTo use the trained model:")
print(f"""
from models.pipeline import FinancialEntitySentimentModel
import torch

model = FinancialEntitySentimentModel(
    encoder_name="{CONFIG['encoder_name']}",
    hidden_size={CONFIG['hidden_size']},
    use_crf_ner={CONFIG['use_crf']},
)
checkpoint = torch.load("checkpoints/stage3_sentiment_large/best_model.pt", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
""")

print(f"\nCheckpoint files:")
!ls -lh "{CONFIG['stage3_checkpoint_dir']}"

In [ ]:
# Copy all local checkpoints to Drive, then terminate
import os, shutil, glob

# Ensure Drive is mounted
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except Exception as e:
    print(f"Drive mount issue: {e}")

drive_ckpt_dir = CONFIG["stage3_checkpoint_dir"]
os.makedirs(drive_ckpt_dir, exist_ok=True)

# Copy ALL local checkpoints to Drive
local_ckpts = glob.glob("/content/*.pt") + glob.glob("/content/checkpoint_epoch_*.pt")
print(f"Found {len(local_ckpts)} local checkpoint files to sync:")
for src in sorted(local_ckpts):
    fname = os.path.basename(src)
    dst = os.path.join(drive_ckpt_dir, fname)
    size_mb = os.path.getsize(src) / 1e6
    print(f"  Copying {fname} ({size_mb:.0f} MB) to Drive...", end=" ")
    try:
        shutil.copy2(src, dst)
        # Verify the copy
        dst_size = os.path.getsize(dst) / 1e6
        if abs(dst_size - size_mb) < 1:
            print(f"OK ({dst_size:.0f} MB)")
        else:
            print(f"WARNING: size mismatch (src={size_mb:.0f}, dst={dst_size:.0f})")
    except Exception as e:
        print(f"FAILED: {e}")

# Flush Drive to ensure writes land
print("\nFlushing Drive...")
try:
    drive.flush_and_unmount()
    print("Drive flushed and unmounted.")
    drive.mount("/content/drive")
    print("Drive remounted.")
except Exception as e:
    print(f"Flush warning: {e}")

# Final verification
print("\nDrive checkpoint verification:")
for f in sorted(os.listdir(drive_ckpt_dir)):
    path = os.path.join(drive_ckpt_dir, f)
    if f.endswith(".pt"):
        print(f"  {f:35s} {os.path.getsize(path)/1e6:8.1f} MB")

print("\n" + "="*60)
print("All checkpoints synced to Drive. Safe to terminate.")
print("="*60)

# NOW terminate — only after everything is saved
# Commented out so you can verify files first. Uncomment to auto-terminate:
# from google.colab import runtime
# runtime.unassign()
